In [1]:
import torch
from modelscope import snapshot_download, AutoTokenizer
from peft import PeftModel
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_dir = "./qwen/Qwen2___5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_dir, device_map=device, torch_dtype=torch.bfloat16)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:

def predict(messages, model, tokenizer):
    model.eval()
    device = "cuda"
    model.to(device)
    # 1. 使用模板生成文本
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    
    # 2. 编码并确保所有 tensor 都在正确的设备上
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    
    # 3. 开启无梯度模式
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=model_inputs.input_ids,
            attention_mask=model_inputs.attention_mask, # 建议显式传入
            max_new_tokens=512,
            eos_token_id=tokenizer.eos_token_id, # 确保生成能正常停止
            pad_token_id=tokenizer.pad_token_id
        )
    
    # 4. 只取新生成的部分
    generated_ids = [
        output_ids[len(input_ids) :]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

lora_adapter_path = "/root/autodl-tmp/output/Qwen2.5-7b-NER-Fixed/checkpoint-782"

tokenizer = AutoTokenizer.from_pretrained(lora_adapter_path, use_fast=False, trust_remote_code=True)
model=PeftModel.from_pretrained(model, model_id=lora_adapter_path)

if __name__ == "__main__":
    # 初始化 SwanLab (如果需要记录)

    text_data = {
        "instruction": """你是一个中医药领域文本实体识别的专家，你需要从给定的句子中提取：方剂, 中医诊断, 中医证候, 其他治疗, 中医治疗, 中药, 西医治疗, 西医诊断, 临床表现, 中医治则 这些实体. 以 json 格式输出, 如 {"entity_text": "当归", "entity_label": "中药"} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出"没有找到任何实体" """, 
        "input": "文本:上火了可以吃点柴胡"
    }
    
    message = [
        {"role": "system", "content": text_data["instruction"]},
        {"role": "user", "content": text_data["input"]}
    ]

    print(f"正在推理输入: {text_data['input']}")
    
    # 执行推理
    result = predict(message, model, tokenizer)
    
    print(f"推理结果: {result}")
    

正在推理输入: 文本:上火了可以吃点柴胡
推理结果: {"entity_text": "柴胡", "entity_label": "中药"}


# 合并lora输出为一个模型

In [13]:
model=PeftModel.from_pretrained(model, model_id=lora_adapter_path)
model = model.merge_and_unload()


output_path = "model/med_ner_qwen7b"
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)


('model/med_ner_qwen7b/tokenizer_config.json',
 'model/med_ner_qwen7b/special_tokens_map.json',
 'model/med_ner_qwen7b/vocab.json',
 'model/med_ner_qwen7b/merges.txt',
 'model/med_ner_qwen7b/added_tokens.json')

In [14]:
if __name__ == "__main__":
    # 初始化 SwanLab (如果需要记录)

    text_data = {
        "instruction": """你是一个中医药领域文本实体识别的专家，你需要从给定的句子中提取：方剂, 中医诊断, 中医证候, 其他治疗, 中医治疗, 中药, 西医治疗, 西医诊断, 临床表现, 中医治则 这些实体. 以 json 格式输出, 如 {"entity_text": "当归", "entity_label": "中药"} 注意: 1. 输出的每一行都必须是正确的 json 字符串. 2. 找不到任何实体时, 输出"没有找到任何实体" """, 
        "input": "文本:上火了可以吃点柴胡"
    }
    
    message = [
        {"role": "system", "content": text_data["instruction"]},
        {"role": "user", "content": text_data["input"]}
    ]

    print(f"正在推理输入: {text_data['input']}")
    
    # 执行推理
    result = predict(message, model, tokenizer)
    
    print(f"推理结果: {result}")

正在推理输入: 文本:上火了可以吃点柴胡
推理结果: {"entity_text": "柴胡", "entity_label": "中药"}
